In [1]:
import ast
import csv
import json
import random
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split


In [2]:
PROJECT_ROOT = Path("C:/Users/Hzaab/SDP-Sequential-Diagnosis/afabench/Afabench_Source/afabench")

dataset = PROJECT_ROOT.parent / "data" / "release_train_patients.csv"
evidences = PROJECT_ROOT.parent / "data" / "release_evidences.json"

output_root= PROJECT_ROOT / "extra" / "data" / "misc"

seed = 42

### load data


In [3]:
def load_raw_patients():
    with open(dataset, newline="", encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))
    for source_row, row in enumerate(rows):
        row["_source_row"] = source_row

    return rows
        

def load_evidence_definitions():
    with open(evidences, encoding="utf-8") as f:
        return json.load(f)

def select_patients(rows):
    return random.Random(seed).sample(rows, N_patients)


def save_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


### feature construction


In [4]:
ordered = ["E_59", "E_56", "E_58", "E_134", "E_136"]


def construct_features(evidence_defs): #binary, multivalued, and ordered/categorical
    features = ["age", "sex", "pain"]

    for eid, d in evidence_defs.items():
        if eid in ("E_53", "E_55"):
            continue

        values = d["possible-values"]

        if d["data_type"] == "B":
            features.append(eid)

        elif d["data_type"] == "M":
            features += [f"{eid}@{v}" for v in values]

        elif d["data_type"] == "C":
            if eid in ordered:
                features += [f"{eid}_level_{v}" for v in values]
            else:
                features += [f"{eid}@{v}" for v in values]

    return sorted(features)

### feature representation

In [5]:
def extract_patient_evidence(row): #get evidence for one patient
    return ast.literal_eval(row["EVIDENCES"])

def pain_is_present(row): #if pain exists yes/no
    for x in extract_patient_evidence(row):
        if x == "E_53" or x.startswith("E_53" + "_@_"):
            return True
        if x == "E_55":
            return True
        if x.startswith("E_55" + "_@_") and not x.endswith("V_123"): #as long as value isnt nowehre
            return True

    return False

def encode_sex(sex):
    return 0.0 if sex.upper() == "F" else 1.0

def encode_row(row, feature_names, evidence_defs): #convert one patient into a feature vector
    values = {x: 0.0 for x in feature_names}

    values["age"] = float(row["AGE"])
    values["sex"] = encode_sex(row["SEX"])
    values["pain"] = float(pain_is_present(row))

    for item in extract_patient_evidence(row):
        evidence_id, _, value_id = item.partition("_@_")

        if evidence_id in ("E_53", "E_55") or evidence_id not in evidence_defs:
            continue

        definition = evidence_defs[evidence_id]

        if definition["data_type"] == "C":
            if evidence_id in ordered:
                name = f"{evidence_id}_level_{value_id}"
            else:
                name = f"{evidence_id}@{value_id}"

        elif definition["data_type"] == "M":
            name = f"{evidence_id}@{value_id}"

        else:
             name = evidence_id

        if name in values:
            values[name] = 1.0
    return [values[x] for x in feature_names]

### save final dataset + visualization

In [ ]:
def build_label_mapping(rows):#convert pathology names to numeric labels
    labels = sorted(set(row["PATHOLOGY"].strip() for row in rows))
    return {label: i for i, label in enumerate(labels)}

def clean_dataframe(rows, feature_names, labels, evidence_defs):
    data = []

    for row in rows:
        initial = row["INITIAL_EVIDENCE"].strip()
        raw_evidence = extract_patient_evidence(row)

        present_ids = {
            item.partition("_@_")[0]
            for item in raw_evidence
        }

        if initial not in present_ids:
            raise ValueError(
                f"Initial evidence {initial} missing from "
                f"source row {row['_source_row']}"
            )

        data.append(
            encode_row(row, feature_names, evidence_defs)
            + [
                labels[row["PATHOLOGY"].strip()],
                row["_source_row"],
                initial,
            ]
        )

    return pd.DataFrame(
        data,
        columns=feature_names + [
            "pathology",
            "source_row",
            "initial_evidence",
        ],
    )


train: 820,481 patients → C:\Users\Hzaab\SDP-Sequential-Diagnosis\afabench\Afabench_Source\afabench\extra\data\misc\ddxplus_train.csv
val: 102,560 patients → C:\Users\Hzaab\SDP-Sequential-Diagnosis\afabench\Afabench_Source\afabench\extra\data\misc\ddxplus_val.csv
test: 102,561 patients → C:\Users\Hzaab\SDP-Sequential-Diagnosis\afabench\Afabench_Source\afabench\extra\data\misc\ddxplus_test.csv
Features: 809
Classes: 49


# Run the below cell for afabench

In [ ]:

rows = load_raw_patients()
evidence_defs = load_evidence_definitions()

feature_names = construct_features(evidence_defs)
labels = build_label_mapping(rows)

df = clean_dataframe(rows, feature_names, labels, evidence_defs)

# 80% training, 20% reserved for validation and testing
train_df, remaining_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["pathology"],
    random_state=seed,
)

# Divide the remaining 20% equally
val_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    stratify=remaining_df["pathology"],
    random_state=seed,
)

output_root.mkdir(parents=True, exist_ok=True)

for name, split in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    path = output_root / f"ddxplus_{name}.csv"
    split.to_csv(path, index=False)
    print(f"{name}: {len(split):,} patients → {path}")

save_json(labels, output_root / "label_mapping.json")

print("Features:", len(feature_names))
print("Classes:", len(labels))

# this is for diaformer:

In [ ]:

N_patients = len(load_raw_patients())
rows = select_patients(load_raw_patients())
evidence_defs = load_evidence_definitions()

feature_names = construct_features(evidence_defs)
labels = build_label_mapping(rows)

df = clean_dataframe(rows, feature_names, labels, evidence_defs)

output_root.mkdir(parents=True, exist_ok=True)

df.to_csv(output_root / "ddxplus_test.csv", index=False)
save_json(labels, output_root / "label_mapping.json")

print("Patients:", len(df))
print("Features:", len(feature_names))
print("Classes:", len(labels))
print("Shape:", df.shape)

print("First 5 patients:")
df.head()